In [1]:
from skimage import feature
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import os
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.neighbors import KNeighborsClassifier as KNN
import xgboost as XGB


In [2]:
parent_folder = Path().resolve().parent
src_path = parent_folder / 'src'
sys.path.append(str(src_path))

from tools import get_embedding_birdnet

#env to use: clef

In [3]:
root_folder='../data/train_data/embedding/birdnet/'

In [4]:
df_pos = get_embedding_birdnet(root_folder, 1)
df_neg = get_embedding_birdnet(root_folder, 0)

In [29]:
df_neg["filename"] = df_neg["embed_name"].str.split("_").str[0]
df_pos["filename"] = df_pos["embed_name"].str.split("-").str[0].str[:-1]

In [30]:
df_pos['target'] = 1
df_neg['target'] = 0

In [31]:
df_neg.sample(10)

,embed_name,embedding,filename,target
445,c0EFhz_1733.birdnet.embeddings.txt,"[0.0, 0.3592633, 0.08421966, 0.2470782, 1.3971...",c0EFhz,0
1213,jlcfzd_539.birdnet.embeddings.txt,"[0.0, 0.158053, 0.3559452, 0.41726694, 0.0, 0....",jlcfzd,0
2079,pZLQMe_1958.birdnet.embeddings.txt,"[0.0, 0.17713504, 0.0, 0.66556835, 0.14271876,...",pZLQMe,0
37,3dBhE6_1308.birdnet.embeddings.txt,"[0.0, 0.6493351, 0.39423093, 1.077464, 0.60584...",3dBhE6,0
713,FKlxjH_926.birdnet.embeddings.txt,"[1.31593, 0.0, 0.0355349, 0.0, 0.0, 0.45997676...",FKlxjH,0
2483,xPBJ0r_867.birdnet.embeddings.txt,"[0.45636395, 0.0, 0.6965607, 0.52779585, 0.0, ...",xPBJ0r,0
488,c0EFhz_492.birdnet.embeddings.txt,"[0.29625568, 0.08919033, 0.15895708, 0.1672854...",c0EFhz,0
325,5da8q2_712.birdnet.embeddings.txt,"[0.114828534, 0.07470498, 0.2275226, 0.310483,...",5da8q2,0
944,IlaTbo_1372.birdnet.embeddings.txt,"[0.0, 0.49257502, 0.614777, 0.20508714, 0.8184...",IlaTbo,0
1596,oOw8he_838.birdnet.embeddings.txt,"[0.0, 0.18106605, 0.0, 0.0, 0.0, 0.14369585, 0...",oOw8he,0


Perform 5-fold split for the negative data only

In [38]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Create a new column to store fold numbers
df_neg["fold"] = -1  # Initialize with -1

for fold, (train_idx, test_idx) in enumerate(kf.split(df_neg)):
    df_neg.loc[test_idx, "fold"] = fold  # Assign fold number to test samples

<IPython.core.display.Javascript object>

In [57]:
fold = 0 # Run the rest of the code for each fold

In [58]:
df_neg_fold = df_neg[df_neg.fold==fold]
df_neg_fold = df_neg_fold.drop("fold", axis=1)

In [59]:
from sklearn.model_selection import GroupKFold, cross_val_score

# Combine and shuffle
df_combined = pd.concat([df_pos, df_neg_fold], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=22).reset_index(drop=True)

# Features, labels, groups
X = np.vstack(df_combined["embedding"].values)
y = df_combined["target"].values
groups = df_combined["filename"].values

# 5-fold grouped CV
cv = GroupKFold(n_splits=5)

SVM

In [60]:
from sklearn.svm import SVC

model = SVC(kernel="rbf", cache_size=500)

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")

ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)


print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9810 ± 0.0089
AP: 0.9988 ± 0.0011


Random Forest

In [61]:
model = RFC(n_jobs = -1)

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")

ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)


print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9680 ± 0.0168
AP: 0.9961 ± 0.0021


XGBoost

In [62]:
model = XGB.XGBClassifier(objective='binary:logistic')

accuracies = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print(f"Accuracy: {accuracies.mean():.4f} ± {accuracies.std(ddof=1):.4f}")

ap = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)


print(f"AP: {ap.mean():.4f} ± {ap.std(ddof=1):.4f}")

Accuracy: 0.9501 ± 0.0255
AP: 0.9893 ± 0.0100


In [64]:
#predictions = model.predict(test)
#ap = average_precision_score(test_target, predictions)
#print("Test set average precision:", ap)

#report=classification_report(test_target, predictions, digits=4)
#print(report)

# Only for binary classification (adjust for multi-class)
#precision, recall, thr = precision_recall_curve(test_target, predictions)


In [60]:
#plt.plot(recall, precision, marker='.')
#plt.xlabel('Recall')
#plt.ylabel('Precision')
#plt.title('Precision-Recall Curve')
#plt.show()